# Implementing a Transformer

## Goal

Build a complete Encoder-Decoder Transformer from scratch in PyTorch, then train it on a simple copy/reverse sequence task to understand how each component works.


1.Setup + config <br/>
2.Positional encoding <br/>
3.Scaled dot-product attention <br/>
4.Multi-head attention <br/>
5.Position-wise feed-forward <br/>
6.Encoder layer (+ Add & Norm) <br/>
7.Decoder layer (+ masked & cross attention) <br/>
8.Full encoder / decoder stacks <br/>
9.The complete Transformer (embeddings + output head) <br/>
10.Masks <br/>
11.The copy/reverse data + training loop <br/>
12.Run it and watch the loss drop <br/>


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

In [ ]:
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
@dataclass
class Config:
  vocab_size: int = 22
  d_model: int = 128
  n_heads: int = 8
  d_ff: int = 512
  n_layers: int = 3
  max_len: int = 64
  dropout: float = 0.1

  @property
  def d_k(self) -> int:
    assert self.d_model % self.n_heads == 0, "d_model must be divisible by n_heads"
    return self.d_model // self.n_heads

cfg = Config()
print(f"d_model={cfg.d_model}, n_heads={cfg.n_heads}, d_k={cfg.d_k}, d_ff={cfg.d_ff}")

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int, dropout: float):
        super().__init__()
        self.dropout = nn.Dropout(dropout)

        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()

        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)

        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        seq_len = x.size(1)
        x = x + self.pe[:, :seq_len]
        return self.dropout(x)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None, dropout=None):
    d_k  = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1))
    scores = scores / math.sqrt(d_k)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    weights = torch.softmax(scores, dim=-1)
    if dropout is not None:
        weights = dropout(weights)

    output = torch.matmul(weights, V)                      # (..., seq_q, d_v)
    return output, weights

In [34]:
torch.manual_seed(0)
q = torch.randn(1, 4, cfg.d_k)
k = torch.randn(1, 4, cfg.d_k)
v = torch.randn(1, 4, cfg.d_k)

causal = torch.tril(torch.ones(4, 4)).unsqueeze(0)
out, w = scaled_dot_product_attention(q, k, v, mask=causal)

print("output shape:", out.shape)
print("weights shape:", w.shape)
print("row sums:", w.sum(dim=-1))
print(w[0].round(decimals=2))

output shape: torch.Size([1, 4, 16])
weights shape: torch.Size([1, 4, 4])
row sums: tensor([[1.0000, 1.0000, 1.0000, 1.0000]])
tensor([[1.0000, 0.0000, 0.0000, 0.0000],
        [0.5100, 0.4900, 0.0000, 0.0000],
        [0.4400, 0.4100, 0.1500, 0.0000],
        [0.1500, 0.2700, 0.4600, 0.1100]])


In [35]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_q = nn.Linear(d_model, d_model, bias=False)
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.W_o = nn.Linear(d_model, d_model, bias=False)  # the final W^O mixing matrix

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key, value, mask=None):
        batch_size = query.size(0)

        Q = self.W_q(query)
        K = self.W_k(key)
        V = self.W_v(value)

        Q = Q.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

        if mask is not None:
            mask = mask.unsqueeze(1)


        out, self.attn_weights = scaled_dot_product_attention(
            Q, K, V, mask=mask, dropout=self.dropout
        )
        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)

        return self.W_o(out)

In [ ]:
class PositionWiseFeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.linear1(x)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

In [ ]:
class EncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_out = self.self_attn(x, x, x, mask=mask)
        x = self.norm1(x + self.dropout1(attn_out))

        ff_out = self.feed_forward(x)
        x = self.norm2(x + self.dropout2(ff_out))

        return x

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.self_attn  = MultiHeadAttention(d_model, n_heads, dropout)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout)
        self.feed_forward = PositionWiseFeedForward(d_model, d_ff, dropout)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, x, enc_out, tgt_mask=None, src_mask=None):
        attn_out = self.self_attn(x, x, x, mask=tgt_mask)
        x = self.norm1(x + self.dropout1(attn_out))
        cross_out = self.cross_attn(x, enc_out, enc_out, mask=src_mask)
        x = self.norm2(x + self.dropout2(cross_out))

        ff_out = self.feed_forward(x)
        x = self.norm3(x + self.dropout3(ff_out))

        return x

In [ ]:
class Encoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

    def forward(self, src, src_mask=None):
        x = self.embed(src) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, mask=src_mask)

        return x

In [ ]:
class Decoder(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_layers, max_len, dropout):
        super().__init__()
        self.d_model = d_model
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos_enc = PositionalEncoding(d_model, max_len, dropout)

        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])

    def forward(self, tgt, enc_out, tgt_mask=None, src_mask=None):
        x = self.embed(tgt) * math.sqrt(self.d_model)
        x = self.pos_enc(x)

        for layer in self.layers:
            x = layer(x, enc_out, tgt_mask=tgt_mask, src_mask=src_mask)

        return x

In [ ]:
class Transformer(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.encoder = Encoder(cfg.vocab_size, cfg.d_model, cfg.n_heads, cfg.d_ff,
                               cfg.n_layers, cfg.max_len, cfg.dropout)
        self.decoder = Decoder(cfg.vocab_size, cfg.d_model, cfg.n_heads, cfg.d_ff,
                               cfg.n_layers, cfg.max_len, cfg.dropout)
        self.output_proj = nn.Linear(cfg.d_model, cfg.vocab_size)

        self._init_parameters()

    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):
        enc_out = self.encoder(src, src_mask=src_mask)
        dec_out = self.decoder(tgt, enc_out,
                               tgt_mask=tgt_mask, src_mask=src_mask)
        logits = self.output_proj(dec_out)
        return logits

In [ ]:
def make_pad_mask(seq, pad_idx):
    mask = (seq != pad_idx)
    return mask.unsqueeze(1).unsqueeze(2)


def make_causal_mask(size, device):
    mask = torch.tril(torch.ones(size, size, device=device, dtype=torch.bool))
    return mask.unsqueeze(0).unsqueeze(0)


def make_masks(src, tgt, pad_idx, device):
    src_mask = make_pad_mask(src, pad_idx)

    tgt_pad = make_pad_mask(tgt, pad_idx)
    tgt_causal = make_causal_mask(tgt.size(1), device)
    tgt_mask = tgt_pad & tgt_causal

    return src_mask, tgt_mask

In [ ]:
PAD_IDX, BOS_IDX, EOS_IDX = 0, 1, 2
DIGIT_OFFSET = 3
NUM_DIGITS = 10

def make_batch(batch_size, min_len=4, max_len=10, device=device):
    src_list, tgt_list = [], []
    for _ in range(batch_size):
        length = torch.randint(min_len, max_len + 1, (1,)).item()
        digits = torch.randint(0, NUM_DIGITS, (length,)) + DIGIT_OFFSET  # random digits

        src_seq = torch.cat([digits, torch.tensor([EOS_IDX])])
        tgt_seq = torch.cat([torch.tensor([BOS_IDX]),
                             torch.flip(digits, dims=[0]),
                             torch.tensor([EOS_IDX])])
        src_list.append(src_seq)
        tgt_list.append(tgt_seq)

    src = nn.utils.rnn.pad_sequence(src_list, batch_first=True, padding_value=PAD_IDX)
    tgt = nn.utils.rnn.pad_sequence(tgt_list, batch_first=True, padding_value=PAD_IDX)
    return src.to(device), tgt.to(device)

src, tgt = make_batch(3)
print("src shape:", src.shape, " tgt shape:", tgt.shape)
print("src[0]:", src[0].tolist())
print("tgt[0]:", tgt[0].tolist())

In [ ]:
def train(model, cfg, steps=2000, batch_size=64, lr=1e-4, log_every=200):
    model.train()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, betas=(0.9, 0.98), eps=1e-9)
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX)

    for step in range(1, steps + 1):
        src, tgt = make_batch(batch_size)

        tgt_in  = tgt[:, :-1]
        tgt_out = tgt[:, 1:]

        src_mask, tgt_mask = make_masks(src, tgt_in, PAD_IDX, device)

        logits = model(src, tgt_in, src_mask=src_mask, tgt_mask=tgt_mask)
        loss = criterion(
            logits.reshape(-1, logits.size(-1)),
            tgt_out.reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        if step % log_every == 0 or step == 1:
            print(f"step {step:4d} | loss {loss.item():.4f}")

    return model

In [ ]:
src, tgt = make_batch(3)
print("src[0]:", src[0].tolist())
print("tgt[0]:", tgt[0].tolist())

src, tgt = make_batch(1)
src_digits = src[0][:-1]
tgt_digits = tgt[0][1:-1]
print("src digits:      ", src_digits.tolist())
print("tgt digits:      ", tgt_digits.tolist())
print("src reversed:    ", torch.flip(src_digits, dims=[0]).tolist())
print("match?", torch.equal(torch.flip(src_digits, dims=[0]), tgt_digits))

In [30]:
model = Transformer(cfg).to(device)

model = train(model, cfg, steps=2000, batch_size=64, lr=1e-4, log_every=200)

step    1 | loss 3.6591
step  200 | loss 1.5822
step  400 | loss 1.2576
step  600 | loss 0.9422
step  800 | loss 0.8506
step 1000 | loss 0.7543
step 1200 | loss 0.5676
step 1400 | loss 0.5001
step 1600 | loss 0.3944
step 1800 | loss 0.3790
step 2000 | loss 0.3007


In [31]:
@torch.no_grad()
def generate(model, src, max_len=20, device=device):
    model.eval()

    src_mask = make_pad_mask(src, PAD_IDX)
    enc_out = model.encoder(src, src_mask=src_mask)

    tgt = torch.tensor([[BOS_IDX]], device=device)

    for _ in range(max_len):
        tgt_mask = make_causal_mask(tgt.size(1), device)

        dec_out = model.decoder(tgt, enc_out, tgt_mask=tgt_mask, src_mask=src_mask)
        logits = model.output_proj(dec_out)


        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)


        tgt = torch.cat([tgt, next_token], dim=1)


        if next_token.item() == EOS_IDX:
            break

    return tgt.squeeze(0)

In [32]:
def decode_tokens(ids):
    """Strip specials, convert token IDs back to readable digits."""
    digits = [t - DIGIT_OFFSET for t in ids
              if t not in (PAD_IDX, BOS_IDX, EOS_IDX)]
    return digits

model.eval()
correct = 0
n_tests = 10
for i in range(n_tests):
    src, tgt = make_batch(1)
    generated = generate(model, src)

    src_digits = decode_tokens(src[0].tolist())
    expected   = src_digits[::-1]
    got        = decode_tokens(generated.tolist())

    ok = (got == expected)
    correct += ok
    if i < 5:  # print the first few
        print(f"src:      {src_digits}")
        print(f"expected: {expected}")
        print(f"got:      {got}   {'CORRECT' if ok else 'WRONG'}")
        print()

print(f"accuracy: {correct}/{n_tests}")

src:      [8, 2, 8, 2, 8, 0, 9, 5, 0]
expected: [0, 5, 9, 0, 8, 2, 8, 2, 8]
got:      [0, 5, 9, 0, 8, 2, 8, 2, 8]   CORRECT

src:      [7, 0, 4, 6, 2, 4]
expected: [4, 2, 6, 4, 0, 7]
got:      [4, 2, 6, 4, 0, 7]   CORRECT

src:      [8, 8, 3, 6, 6, 1, 7, 2, 6, 7]
expected: [7, 6, 2, 7, 1, 6, 6, 3, 8, 8]
got:      [7, 6, 2, 7, 1, 6, 6, 3, 8, 8]   CORRECT

src:      [3, 8, 9, 2, 9]
expected: [9, 2, 9, 8, 3]
got:      [9, 2, 9, 8, 3]   CORRECT

src:      [7, 5, 2, 7, 4, 3, 2, 3, 2]
expected: [2, 3, 2, 3, 4, 7, 2, 5, 7]
got:      [2, 3, 2, 3, 4, 7, 2, 5, 7]   CORRECT

accuracy: 10/10


In [33]:
torch.save({
    'model_state_dict': model.state_dict(),
    'config': cfg,
}, 'transformer_reverse.pt')
print("Saved to transformer_reverse.pt")

Saved to transformer_reverse.pt
